# Chronos-2 Inference Speed Profiling

Benchmarks two inference paths:
- **Baseline** — `Chronos2Pipeline.predict()`: the standard high-level API with full dataset/DataLoader machinery
- **Fast** — `Chronos2Pipeline._predict_step()`: pre-built GPU tensors, skipping dataset construction and DataLoader overhead

Both paths exercise the same underlying `Chronos2Model.forward()` call.

## 1. Config

In [1]:
import math
import time
import warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
from torch.utils.data import DataLoader

from chop.models.chronos2.pipeline import Chronos2Pipeline
from chronos.chronos2.dataset import Chronos2Dataset, DatasetMode

# ── Config ────────────────────────────────────────────────────────────────────
DEVICE           = 'cuda' if torch.cuda.is_available() else 'cpu'
SEQ_LEN          = 1440   # context length per time series
N_FEATURES       = 20     # number of variates per sample
FORECAST_HORIZON = 96     # prediction length
BATCH_SIZES      = [20, 40, 60, 80, 100]  # number of multivariate samples per batch
WARMUP_RUNS      = 3
BENCH_RUNS       = 5
MODEL_ID         = 'amazon/chronos-2'

print(f'Device: {DEVICE}')
print(f'Sequence length: {SEQ_LEN}')
print(f'Number of features: {N_FEATURES}')
print(f'Forecast horizon: {FORECAST_HORIZON}')

Device: cuda
Sequence length: 1440
Number of features: 20
Forecast horizon: 96


## 2. Load model

In [2]:
pipeline = Chronos2Pipeline.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    torch_dtype=torch.bfloat16,
)
model = pipeline.model
model.eval()

OUTPUT_PATCH_SIZE = model.chronos_config.output_patch_size
NUM_OUTPUT_PATCHES = math.ceil(FORECAST_HORIZON / OUTPUT_PATCH_SIZE)
FUTURE_LEN = NUM_OUTPUT_PATCHES * OUTPUT_PATCH_SIZE

print(f'✓ Model loaded successfully from {MODEL_ID}')
print(f'  Model type:        {type(model).__name__}')
print(f'  Output patch size: {OUTPUT_PATCH_SIZE}')
print(f'  Num output patches for horizon={FORECAST_HORIZON}: {NUM_OUTPUT_PATCHES}')

✓ Model loaded successfully from amazon/chronos-2
  Model type:        Chronos2Model
  Output patch size: 16
  Num output patches for horizon=96: 6


## 3. Verify identical results

Run both paths on the same small input and confirm they produce the same predictions.

In [3]:
torch.manual_seed(42)
VERIFY_BATCH = 2  # small batch for verification

# Inputs must be list-of-dicts with "target" key
verify_inputs = [{"target": torch.randn(N_FEATURES, SEQ_LEN)} for _ in range(VERIFY_BATCH)]

# ── Baseline: pipeline.predict() ─────────────────────────────────────────────
with torch.no_grad():
    baseline_preds = pipeline.predict(
        verify_inputs,
        prediction_length=FORECAST_HORIZON,
        batch_size=VERIFY_BATCH * N_FEATURES,
    )  # list of (N_FEATURES, n_quantiles, FORECAST_HORIZON) tensors

# ── Fast: extract pre-processed batch from Chronos2Dataset, call _predict_step directly ──
verify_dataset = Chronos2Dataset.convert_inputs(
    verify_inputs,
    context_length=model.chronos_config.context_length,
    prediction_length=FORECAST_HORIZON,
    batch_size=VERIFY_BATCH * N_FEATURES,
    output_patch_size=OUTPUT_PATCH_SIZE,
    mode=DatasetMode.TEST,
)
verify_loader = DataLoader(verify_dataset, batch_size=None, shuffle=False)
verify_batch  = next(iter(verify_loader))

v_context  = verify_batch['context'].to(device=DEVICE, dtype=torch.float32)
v_gids     = verify_batch['group_ids'].to(device=DEVICE)
v_fut_cov  = verify_batch['future_covariates'].to(device=DEVICE, dtype=torch.float32)
v_ranges   = verify_batch['target_idx_ranges']

with torch.no_grad():
    fast_raw = pipeline._predict_step(
        context=v_context,
        group_ids=v_gids,
        future_covariates=v_fut_cov,
        num_output_patches=NUM_OUTPUT_PATCHES,
    )  # (total_series, n_quantiles, FUTURE_LEN)

# Slice to per-sample predictions to match baseline output format
fast_preds = [fast_raw[start:end, :, :FORECAST_HORIZON].cpu() for (start, end) in v_ranges]

# Compare
all_close = all(
    torch.allclose(b.float(), f.float(), atol=1e-3)
    for b, f in zip(baseline_preds, fast_preds)
)
if all_close:
    print('✓ Baseline and fast implementations produce identical results')
else:
    max_diff = max(
        (b.float() - f.float()).abs().max().item()
        for b, f in zip(baseline_preds, fast_preds)
    )
    print(f'✗ Results differ (max absolute diff: {max_diff:.6f})')

✓ Baseline and fast implementations produce identical results


## 4. Benchmark helpers

In [4]:
def sync():
    if DEVICE == 'cuda':
        torch.cuda.synchronize()


def benchmark_baseline(batch_size: int) -> float:
    """
    Standard pipeline.predict() — full dataset + DataLoader machinery.
    Returns seconds-per-sample.
    """
    inputs = [{"target": torch.randn(N_FEATURES, SEQ_LEN)} for _ in range(batch_size)]

    # warmup
    for _ in range(WARMUP_RUNS):
        with torch.no_grad():
            pipeline.predict(
                inputs,
                prediction_length=FORECAST_HORIZON,
                batch_size=batch_size * N_FEATURES,
            )

    sync()
    t0 = time.perf_counter()
    for _ in range(BENCH_RUNS):
        with torch.no_grad():
            pipeline.predict(
                inputs,
                prediction_length=FORECAST_HORIZON,
                batch_size=batch_size * N_FEATURES,
            )
    sync()
    elapsed = (time.perf_counter() - t0) / BENCH_RUNS
    return elapsed / batch_size


def benchmark_fast(batch_size: int) -> float:
    """
    Pre-built GPU tensors → _predict_step() directly, no dataset/DataLoader overhead.
    Returns seconds-per-sample.
    """
    # Pre-build once, not counted in timing
    total_series = batch_size * N_FEATURES
    context = torch.randn(
        total_series, SEQ_LEN, device=DEVICE, dtype=torch.bfloat16
    ).to(torch.float32)
    # group_ids: series 0..N_FEATURES-1 → group 0, N_FEATURES..2*N_FEATURES-1 → group 1, …
    group_ids = torch.arange(batch_size, device=DEVICE).repeat_interleave(N_FEATURES)
    future_covariates = torch.full(
        (total_series, FUTURE_LEN), fill_value=float('nan'), device=DEVICE, dtype=torch.float32
    )

    # warmup
    for _ in range(WARMUP_RUNS):
        with torch.no_grad():
            pipeline._predict_step(
                context=context,
                group_ids=group_ids,
                future_covariates=future_covariates,
                num_output_patches=NUM_OUTPUT_PATCHES,
            )

    sync()
    t0 = time.perf_counter()
    for _ in range(BENCH_RUNS):
        with torch.no_grad():
            pipeline._predict_step(
                context=context,
                group_ids=group_ids,
                future_covariates=future_covariates,
                num_output_patches=NUM_OUTPUT_PATCHES,
            )
    sync()
    elapsed = (time.perf_counter() - t0) / BENCH_RUNS
    return elapsed / batch_size

## 5. Run benchmarks

In [ ]:
baseline_results = {}
fast_results     = {}

print('=== Baseline Implementation Benchmark ===')
for bs in BATCH_SIZES:
    sps = benchmark_baseline(bs)
    baseline_results[bs] = sps
    print(f'  Batch size {bs:3d}: {sps:.4f} s/sample')

print()
print('=== Fast Implementation Benchmark ===')
for bs in BATCH_SIZES:
    sps = benchmark_fast(bs)
    fast_results[bs] = sps
    print(f'  Batch size {bs:3d}: {sps:.4f} s/sample')

=== Baseline Implementation Benchmark ===
  Batch size  20: 0.1684 s/sample


## 6. Visualise results

In [ ]:
import matplotlib.pyplot as plt

xs      = list(baseline_results.keys())
base_ys = [baseline_results[b] * 1000 for b in xs]  # ms/sample
fast_ys = [fast_results[b]     * 1000 for b in xs]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: latency curves
ax = axes[0]
ax.plot(xs, base_ys, 'o-', color='steelblue', label='Baseline (pipeline.predict)')
ax.plot(xs, fast_ys, 's-', color='coral',     label='Fast (_predict_step)')
ax.set_xlabel('Batch size (number of samples)')
ax.set_ylabel('Latency (ms / sample)')
ax.set_title('Inference latency per sample')
ax.legend()
ax.grid(True, alpha=0.3)

# Right: speedup ratio
ax2 = axes[1]
speedups = [b / f for b, f in zip(base_ys, fast_ys)]
ax2.bar(xs, speedups, color='mediumseagreen', width=8)
ax2.axhline(1.0, color='k', linestyle='--', linewidth=0.8)
ax2.set_xlabel('Batch size')
ax2.set_ylabel('Speedup (baseline / fast)')
ax2.set_title('Fast vs Baseline speedup')
ax2.grid(True, alpha=0.3, axis='y')

fig.suptitle(
    f'Chronos-2 Inference Speed  |  device={DEVICE}  |  seq={SEQ_LEN}  '
    f'features={N_FEATURES}  horizon={FORECAST_HORIZON}',
    fontsize=10,
)
fig.tight_layout()
plt.savefig('artifacts/speed_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved speed_benchmark.png')

## 7. Throughput summary

In [ ]:
print('=' * 60)
print('CHRONOS-2 SPEED PROFILING SUMMARY')
print('=' * 60)
print(f'  Device:           {DEVICE}')
print(f'  Sequence length:  {SEQ_LEN}')
print(f'  Features:         {N_FEATURES}')
print(f'  Horizon:          {FORECAST_HORIZON}')
print(f'  Warmup / bench runs: {WARMUP_RUNS} / {BENCH_RUNS}')
print()
print(f'{"Batch":>8}  {"Baseline (ms/s)":>18}  {"Fast (ms/s)":>14}  {"Speedup":>9}')
print('-' * 60)
for bs in BATCH_SIZES:
    b = baseline_results[bs] * 1000
    f = fast_results[bs]     * 1000
    print(f'{bs:>8d}  {b:>18.2f}  {f:>14.2f}  {b/f:>8.2f}x')
print('=' * 60)